In [ ]:
%run ../utils/logger

In [ ]:
%run ../utils/dataquality

In [ ]:
log_event("gold", "fact_sales", "START", "Building fact table")

query = """
SELECT
    sd.order_number,
    pr.product_key,
    cu.customer_key,
    sd.order_date,
    sd.ship_date,
    sd.due_date,
    sd.sales_amount,
    sd.quantity,
    sd.price
FROM silver.crm_sales sd
LEFT JOIN gold.dim_products pr
    ON sd.product_number = pr.product_number
LEFT JOIN gold.dim_customers cu
    ON sd.customer_id = cu.customer_id
"""
df = spark.sql(query)

In [ ]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.fact_sales")
row_count = check_not_empty(df, "gold.fact_sales")
log_event("gold", "fact_sales", "SUCCESS", "Write complete", rows=row_count)

In [ ]:
check_null_rate(df, "gold.fact_sales", "order_number")
check_null_rate(df, "gold.fact_sales", "product_key")
check_null_rate(df, "gold.fact_sales", "customer_key")
check_duplicates(df, ["order_number", "product_key"])